In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

from utils import (
    objectives_retarget_energies,
    read_fit,
    read_xrr,
)
from utils.graded_figures import (
    bookended_profile_arrays,
)
from utils.graded_objective import extract_graded_fit_context
from utils.helpers.fitting_helper import rxr
from utils.helpers.plotting_helper import set_plotting_defaults
from utils.models import configure_refloxide_fitting

configure_refloxide_fitting()
set_plotting_defaults()

ENERGY = 283.7  # eV

In [ ]:
def resolve_data_key(energy: float, fit_bundle):
    """Return the objective in a saved multi-energy fit at ``energy`` (eV)."""
    return next(o for o in fit_bundle.objectives if o.model.energy == float(energy))

In [ ]:
data = read_xrr("reflectivity_data", material="znpc", source="hub")

graded_fit = read_fit("graded/graded_fit.pkl", material="znpc", source="local")
dft_fit = read_fit("dft/dft_en_offset_new2.pkl", material="znpc", source="local")
free_fit = read_fit("free/free_en_offset_init_2.pkl", material="znpc", source="local")

# Graded model retargeted to the probe energy (rebuilds dispersive OOC caches).
graded_obj = objectives_retarget_energies(graded_fit, [ENERGY], data=data)[ENERGY]


def pick_energy_objective(global_obj, energy):
    """Return the single-energy objective nearest `energy` from a GlobalObjective."""
    energies = np.array([o.model.energy for o in global_obj.objectives], dtype=float)
    idx = int(np.argmin(np.abs(energies - energy)))
    return global_obj.objectives[idx]


dft_obj = pick_energy_objective(dft_fit, ENERGY)
free_obj = pick_energy_objective(free_fit, ENERGY)

# Shared dataset (identical q grid for every model comparison).
dataset = data[str(ENERGY)]

In [ ]:
def nevot_croce_density(film, *, n_points: int = 2000, n_sigma: int = 5):
    """Convolve the book-ended density with the surface (vacuum) Névot-Croce term.

    The Névot-Croce interface is an error function, so its real-space density is
    the analytic profile convolved with a Gaussian of standard deviation equal to
    ``film.surface_roughness``. Vacuum (zero density) bounds the surface side and
    the film-edge density bounds the substrate side so the convolution sees the
    physical boundary on each end.

    Parameters
    ----------
    film
        Book-ended film component.
    n_points
        Samples across the film thickness.
    n_sigma
        Padding width in units of the surface roughness.

    Returns
    -------
    depth, rho_sharp, rho_rough
        Depth (angstrom) and the analytic and roughness-smeared densities
        (g/cm^3) on ``[0, total_thick]``.
    """
    total = float(film.total_thick.value)
    sigma = float(film.surface_roughness.value)
    depth = np.linspace(0.0, total, n_points)
    rho_sharp = np.asarray(film.local_density(depth), dtype=float)
    if sigma <= 0.0:
        return depth, rho_sharp, rho_sharp.copy()

    dz = depth[1] - depth[0]
    pad = int(np.ceil(n_sigma * sigma / dz))
    z_ext = np.concatenate(
        [depth[0] + dz * np.arange(-pad, 0), depth, depth[-1] + dz * np.arange(1, pad + 1)]
    )
    rho_ext = np.empty_like(z_ext)
    below, above = z_ext < 0.0, z_ext > total
    inside = ~(below | above)
    rho_ext[below] = 0.0  # vacuum
    rho_ext[inside] = np.asarray(film.local_density(z_ext[inside]), dtype=float)
    rho_ext[above] = float(film.local_density(total))  # film/substrate edge

    kernel = np.exp(-0.5 * (dz * np.arange(-pad, pad + 1) / sigma) ** 2)
    kernel /= kernel.sum()
    rho_rough = np.convolve(rho_ext, kernel, mode="same")[pad : pad + n_points]
    return depth, rho_sharp, rho_rough

In [ ]:
def model_reflectivity(model, q, pol):
    """Evaluate model reflectivity at a single polarization without mutating state.

    Parameters
    ----------
    model
        Pyref ``ReflectModel`` (refloxide-backed).
    q
        1-D array of momentum transfer in inverse angstrom.
    pol
        Polarization channel, ``"s"`` or ``"p"``.

    Returns
    -------
    numpy.ndarray
        Reflectivity on the requested q grid.
    """
    original = model.pol
    model.pol = pol
    try:
        return np.asarray(model(q), dtype=float)
    finally:
        model.pol = original

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = [
    {"name": "Graded",    "obj": graded_obj, "color": "r", "ls": "-",  "lw": 1.3, "z": 6},
    {"name": "DFT slab",  "obj": dft_obj,    "color": "k", "ls": "--", "lw": 1.0, "z": 4},
    {"name": "Free slab", "obj": free_obj,   "color": "k", "ls": "-",  "lw": 1.0, "z": 4},
]
c_alpha, c_rho = "#3b4cc0", "#e8852b"

cmp = plt.colormaps["tab20"]
c_s, c_p = cmp(1), cmp(0)

qmin = 0.1
q = np.linspace(qmin, float(dataset.s.x.max()), 1500)


def rms_residual(model, pol=("s", "p")):
    """RMS normalized residual over the plotted q window for the given channels."""
    chis = []
    for p in pol:
        d = getattr(dataset, p)
        m = d.x >= qmin
        chis.append((d.y[m] - rxr(d.x[m], model, p)) / d.y_err[m])
    return float(np.sqrt(np.mean(np.concatenate(chis) ** 2)))


# --- nested layout: left = reflectivity + residual strip, right = orientation, density ---
fig = plt.figure(figsize=(3.5, 3.5), dpi=300)
gs_outer = fig.add_gridspec(1, 2, width_ratios=[1.55, 1.0], wspace=0.5)
gs_left = gs_outer[0].subgridspec(2, 1, height_ratios=[3.0, 1.0], hspace=0.05)
gs_right = gs_outer[1].subgridspec(2, 1, hspace=0.22)

ax_R = fig.add_subplot(gs_left[0])
ax_res = fig.add_subplot(gs_left[1], sharex=ax_R)
ax_a = fig.add_subplot(gs_right[0])
ax_d = fig.add_subplot(gs_right[1], sharex=ax_a)

# ---- (a) reflectivity at 283.7 eV ----
for pol, marker in (("s", "o"), ("p", "^")):
    d = getattr(dataset, pol)
    m = d.x >= qmin
    ax_R.errorbar(
        d.x[m], d.y[m], d.y_err[m],
        marker=marker, lw=0, ms=2, elinewidth=0.5, capsize=0,
        color=c_s if pol == "s" else c_p, ecolor="0.7", zorder=2,
    )

for mdl in models:
    rms = rms_residual(mdl["obj"].model)
    for pol in ("s", "p"):
        ax_R.plot(
            q, rxr(q, mdl["obj"].model, pol),
            color=mdl["color"], ls=mdl["ls"], lw=mdl["lw"], zorder=mdl["z"],
            label=f'{mdl["name"]}  {rms:.1f}' if pol == "s" else None,
        )

ax_R.set_yscale("log")
ax_R.set_xlim(qmin, q.max())
ax_R.set_ylim(None, 1e-3)
ax_R.set_ylabel("Reflectivity (a.u.)")
ax_R.tick_params(labelbottom=False)
ax_R.minorticks_on()
ax_R.legend(
    loc="upper right", frameon=False, handlelength=1.8,
    labelspacing=0.25, handletextpad=0.5, borderaxespad=0.2,
    title=r"model   $\chi_\mathrm{rms}$", title_fontsize=8, fontsize=8,
)

# ---- residual strip (p-pol normalized residual per model) ----
ax_res.axhline(0.0, color=plt.rcParams["axes.edgecolor"], lw=plt.rcParams["axes.linewidth"])
d = dataset.p
m = d.x >= qmin
for mdl in models:
    chi = (d.y[m] - rxr(d.x[m], mdl["obj"].model, "p")) / d.y_err[m]
    ax_res.plot(d.x[m], chi, color=mdl["color"], ls=mdl["ls"], lw=0.6, zorder=mdl["z"])
ax_res.set_ylim(-6, 6)
ax_res.set_yticks([-4, 0, 4])
ax_res.set_ylabel(r"$\chi_p$")
ax_res.set_xlabel(r"$q\ (\mathrm{\AA}^{-1})$")
ax_res.minorticks_on()

# ---- profiles (graded film only) ----
film = extract_graded_fit_context(graded_obj).film
total = float(film.total_thick.value)
tau_vac, tau_si = float(film.tau_vac.value), float(film.tau_si.value)
depth, rho_sharp, rho_rough = nevot_croce_density(film)
alpha_deg = np.degrees(np.asarray(film.orientation(depth), dtype=float))

# (data, label, horizontal anchor, x-offset in points) so neither label leaves the axes
tau_marks = (
    (tau_vac, r"$\tau_\mathrm{vac}$", "left", 3),
    (total - tau_si, r"$\tau_\mathrm{Si}$", "right", -3),
)

# ---- (b) orientation ----
ax_a.plot(depth, alpha_deg, color=c_alpha, lw=1.6)
for z_tau, txt, ha, dx in tau_marks:
    ax_a.axvline(z_tau, color="0.4", lw=0.8, ls=(0, (4, 3)), zorder=1)
    ax_a.annotate(txt, xy=(z_tau, 0.97), xycoords=("data", "axes fraction"),
                  xytext=(dx, 0), textcoords="offset points",
                  va="top", ha=ha, color="0.3", fontsize=8)
ax_a.set_ylabel(r"$\alpha$ (deg)")
ax_a.set_ylim(20, 75)
ax_a.set_xlim(0, total)
ax_a.tick_params(labelbottom=False)
ax_a.tick_params(axis="y", labelsize=9)
ax_a.minorticks_on()

# ---- (c) density (Névot-Croce smeared, sharp dotted reference) ----
ax_d.plot(depth, rho_sharp, color=c_rho, lw=0.9, ls=":")
ax_d.plot(depth, rho_rough, color=c_rho, lw=1.6)
for z_tau, *_ in tau_marks:
    ax_d.axvline(z_tau, color="0.4", lw=0.8, ls=(0, (4, 3)), zorder=1)
ax_d.set_ylabel(r"$\rho$ (g cm$^{-3}$)")
ax_d.set_xlabel(r"depth $z\ (\mathrm{\AA})$")
ax_d.set_ylim(None, 2)
ax_d.set_yticks([1, 2])
ax_d.set_xlim(0, total)
ax_d.tick_params(axis="y", labelsize=9)
ax_d.minorticks_on()

# ---- panel labels: (a) in the gutter, (b)/(c) above their narrow axes ----
ax_R.text(-0.34, 1.04, "(a)", transform=ax_R.transAxes, va="bottom", ha="left")
ax_a.text(0.0, 1.06, "(b)", transform=ax_a.transAxes, va="bottom", ha="left")
ax_d.text(0.0, 1.06, "(c)", transform=ax_d.transAxes, va="bottom", ha="left")

fig.align_ylabels([ax_R, ax_res])
fig.savefig("graded_fit_comparison.png", dpi=300, bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
#  print the graded parameters
print(graded_obj.varying_parameters())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from scipy.special import erf

# model identity is carried by COLOR across every panel (fixes cross-panel color break)
C_GRADED, C_DFT, C_FREE = "r", "k", "k"
MODELS = [
    {"name": "Graded",    "obj": graded_obj, "color": C_GRADED, "ls": "-",  "lw": 1, "z": 5},
    {"name": "DFT slab",  "obj": dft_obj,    "color": C_DFT,    "ls": "--", "lw": 1, "z": 4},
    {"name": "Free slab", "obj": free_obj,   "color": C_FREE,   "ls": "-",  "lw": 1, "z": 4},
]
cmp = plt.colormaps["tab20"]
C_S, C_P = cmp(1), cmp(0)

QMIN = 0.1
Q = np.linspace(QMIN, float(dataset.s.x.max()), 1500)


def rms_residual(model, pol=("s", "p")):
    chis = []
    for p in pol:
        d = getattr(dataset, p); m = d.x >= QMIN
        chis.append((d.y[m] - rxr(d.x[m], model, p)) / d.y_err[m])
    return float(np.sqrt(np.mean(np.concatenate(chis) ** 2)))


def slab_film_profiles(model, idx=(1, 2, 3), n=600):
    """Erf-smoothed orientation (deg) and density for the 3-slab ZnPc film.

    Measured from the vacuum interface so the depth axis matches the graded film.
    Assumes vacuum|surf|bulk|inter|SiO2|Si and that component[i].rough sets the
    interface above slab i. Swap r1 and r2 if your roughness convention differs.
    """
    s = [model.structure[i] for i in idx]
    t = np.array([float(c.thick.value) for c in s])
    a = np.array([float(c.sld.rotation.value) for c in s])
    rho = np.array([float(c.sld.density.value) for c in s])
    r1 = max(float(s[1].rough.value), 1e-6)
    r2 = max(float(s[2].rough.value), 1e-6)
    total = float(t.sum())
    z = np.linspace(0.0, total, n)
    z1, z2 = t[0], t[0] + t[1]
    step1 = 0.5 * (1.0 + erf((z - z1) / (np.sqrt(2.0) * r1)))
    step2 = 0.5 * (1.0 + erf((z - z2) / (np.sqrt(2.0) * r2)))
    blend = lambda v: v[0] + (v[1] - v[0]) * step1 + (v[2] - v[1]) * step2
    return z, np.degrees(blend(a)), blend(rho)


# graded profiles (nevot_croce_density and extract_graded_fit_context from earlier cells)
film = extract_graded_fit_context(graded_obj).film
TOTAL = float(film.total_thick.value)
TAU_VAC, TAU_SI = float(film.tau_vac.value), float(film.tau_si.value)
Z_G, RHO_SHARP, RHO_ROUGH = nevot_croce_density(film)
ALPHA_G = np.degrees(np.asarray(film.orientation(Z_G), dtype=float))

# slab profiles in the same depth frame
Z_DFT, ALPHA_DFT, RHO_DFT = slab_film_profiles(dft_obj.model)
Z_FREE, ALPHA_FREE, RHO_FREE = slab_film_profiles(free_obj.model)

RMS = {m["name"]: rms_residual(m["obj"].model) for m in MODELS}


def tag(ax, s, frac = -.4):
    ax.text(frac, .9, s, transform=ax.transAxes, va="bottom", ha="left")


# ---- reusable panel painters (identical content across the three layouts) ----
def draw_reflectivity(ax, legend_loc="lower left"):
    for pol, marker in (("s", "o"), ("p", "p")):
        d = getattr(dataset, pol)
        m = d.x >= QMIN
        ax.errorbar(d.x[m], d.y[m], d.y_err[m], marker=marker, lw=0, ms=2,
                    elinewidth=0.5, capsize=0, color=C_S if pol == "s" else C_P,
                    ecolor=C_S if pol == "s" else C_P, zorder=2, markevery=2)
    for mdl in MODELS:
        for pol in ("s", "p"):
            ax.plot(Q, rxr(Q, mdl["obj"].model, pol), color=mdl["color"], ls=mdl["ls"],
                    lw=mdl["lw"], zorder=mdl["z"],
                    label=f'{mdl["name"]}' if pol == "s" else None)
    for pol, txt, off in (("s", "s", 8), ("p", "p", -20)):  # label the two polarization branches
        d = getattr(dataset, pol)
        i = int(np.argmin(np.abs(d.x - 0.12)))
        ax.annotate(txt, (d.x[i], d.y[i]), textcoords="offset points", xytext=(0, off),
                    ha="center", color="0.3")
    ax.set_yscale("log")
    ax.set_xlim(QMIN, Q.max())
    ax.set_ylim(None, 1e-2)
    ax.set_ylabel("Reflectivity")  # not a.u., reflectivity is a normalized ratio
    ax.legend(loc=legend_loc, handlelength=1)


def draw_residual(ax):
    ax.axhline(0.0, color=plt.rcParams["axes.edgecolor"], lw=plt.rcParams["axes.linewidth"])
    d = dataset.p
    m = d.x >= QMIN
    for mdl in MODELS:
        chi = (d.y[m] - rxr(d.x[m], mdl["obj"].model, "p")) / d.y_err[m]
        ax.plot(d.x[m], chi, color=mdl["color"], ls=mdl["ls"], lw=0.6, zorder=mdl["z"],
                label=f'{RMS[mdl["name"]]:.2g}')
    ax.set_ylim(-6, 6)
    ax.set_yticks([-4, 0, 4])
    ax.set_ylabel(r"$\chi_p$")
    ax.set_xlim(QMIN, Q.max())
    ax.set_xlabel(r"$q\ (\mathrm{\AA}^{-1})$")
    # Directly annotate each model's chi value at the endpoint of its curve
    d = dataset.p
    m = d.x >= QMIN
    for mdl in MODELS:
        chi = (d.y[m] - rxr(d.x[m], mdl["obj"].model, "p")) / d.y_err[m]
        max_idx = np.argmax(np.abs(chi))
        x_pos = d.x[m][max_idx] if mdl["name"] != "Free slab" else d.x[m][max_idx-3]
        y_pos = chi[max_idx]
        ax.annotate(f"{RMS[mdl['name']]:.2g}",
                    xy=(x_pos, y_pos),
                    xytext=(3, 0),
                    textcoords="offset points",
                    va="center", ha="left" if mdl["name"] != "Free slab" else "right",
                    color=mdl["color"], fontsize=7)


def draw_orientation(ax, tau_labels=True):
    ax.plot(Z_FREE, ALPHA_FREE, color=C_FREE, ls="-", lw=0.9, zorder=3)
    ax.plot(Z_DFT, ALPHA_DFT, color=C_DFT, ls="--", lw=0.9, zorder=3)
    ax.plot(Z_G, ALPHA_G, color=C_GRADED, lw=1, zorder=5)
    for z_tau, txt, ha, dx in ((TAU_VAC, r"$\tau_\mathrm{vac}$", "left", 3),
                               (TOTAL - TAU_SI, r"$\tau_\mathrm{Si}$", "right", -3)):
        ax.axvline(z_tau, color="0.55", lw=0.7, ls=(0, (4, 3)), zorder=1)
        if tau_labels:
            ax.annotate(txt, xy=(z_tau, 0.5), xycoords=("data", "axes fraction"),
                        xytext=(dx, 0), textcoords="offset points", va="top", ha=ha, color="0.4")
    ax.set_ylabel(r"$\gamma$ (deg)")
    ax.set_ylim(20, 75)
    ax.set_xlim(0, TOTAL)


def draw_density(ax, key=True):
    ax.plot(Z_FREE, RHO_FREE, color=C_FREE, ls="-", lw=0.9, zorder=3)
    ax.plot(Z_DFT, RHO_DFT, color=C_DFT, ls="--", lw=0.9, zorder=3, label="DFT")
    ax.plot(Z_G, RHO_SHARP, color=C_GRADED, ls=":", lw=0.8, zorder=4, label="Graded")
    ax.plot(Z_G, RHO_ROUGH, color=C_GRADED, lw=1.0, zorder=5, label="Graded")
    for z_tau in (TAU_VAC, TOTAL - TAU_SI):
        ax.axvline(z_tau, color="0.55", lw=0.7, ls=(0, (4, 3)), zorder=1)
    ax.set_ylabel(r"$\rho$ (g cm$^{-3}$)")
    ax.set_xlim(0, TOTAL)
    ax.set_ylim(.5, 2.2)
    ax.set_yticks([1, 2])
    # if key:
    #     ax.legend(handlelength=.5)

In [ ]:
# ---------------------------------------------------------------------------
# Tilt-angle reference markers and the density-weighted PXR orientation average
# Paste below the existing painter definitions; draw_orientation replacement at end.
# ---------------------------------------------------------------------------
import numpy as np


def density_weighted_tilt(z, alpha_deg, rho, weight="cos2"):
    """Density-weighted orientation average over the film.

    weight="cos2" returns arccos(sqrt(<cos^2 gamma>_rho)), the second-moment
    effective tilt. This is the quantity NEXAFS reports, so it is the
    apples-to-apples comparator. weight="linear" returns the plain <gamma>_rho.
    """
    rho = np.clip(np.asarray(rho, dtype=float), 0.0, None)
    norm = np.trapezoid(rho, z)
    if weight == "cos2":
        c2 = np.trapezoid(rho * np.cos(np.radians(alpha_deg)) ** 2, z) / norm
        return float(np.degrees(np.arccos(np.sqrt(c2))))
    return float(np.trapezoid(rho * alpha_deg, z) / norm)


def graded_profiles_live():
    """Re-extract graded film profiles from the CURRENT parameter values.

    Must rebuild from the objective each call so the covariance propagation
    below sees the parameter perturbations.
    """
    film = extract_graded_fit_context(graded_obj).film
    z, _, rho_rough = nevot_croce_density(film)
    alpha = np.degrees(np.asarray(film.orientation(z), dtype=float))
    return z, alpha, rho_rough


def tilt_with_stderr(objective, profile_fn, weight="cos2", rel_step=1e-4):
    """Central value and 1-sigma error by linear propagation of the fit covariance.

    Central-difference Jacobian over the varying parameters, sigma^2 = J C J^T.
    First-order only; the cos^2 average is nonlinear in the orientation
    parameters, so for the manuscript number tilt_posterior() below is the
    more defensible quote if a chain is stored.
    """
    varying = objective.varying_parameters()
    p0 = np.array([float(p.value) for p in varying])

    def f():
        return density_weighted_tilt(*profile_fn(), weight=weight)

    f0 = f()
    J = np.zeros(p0.size)
    for i, p in enumerate(varying):
        h = rel_step * max(abs(p0[i]), 1e-2)
        p.value = p0[i] + h
        fp = f()
        p.value = p0[i] - h
        fm = f()
        p.value = p0[i]
        J[i] = (fp - fm) / (2.0 * h)
    sigma = float(np.sqrt(J @ objective.covar() @ J))
    return f0, sigma


def tilt_posterior(objective, profile_fn, weight="cos2", ngen=300):
    """Posterior spread of the average. Requires a stored MCMC chain.

    Returns (median, err_lo, err_hi) from the 16/50/84 percentiles, ready to
    pass as err=(err_lo, err_hi) to the marker below.
    """
    saved = np.array(objective.parameters.pvals)
    vals = []
    for pvals in objective.pgen(ngen=ngen):
        objective.setp(pvals)
        vals.append(density_weighted_tilt(*profile_fn(), weight=weight))
    objective.setp(saved)
    lo, med, hi = np.percentile(vals, [16, 50, 84])
    return float(med), float(med - lo), float(hi - med)


# ---- marker painter --------------------------------------------------------
def draw_tilt_marker(ax, value, err=None, label="", color="0.25",
                     style="tick", show_err=True, tick_frac=0.14,
                     ls=(0, (1, 1.5)), fontsize=7):
    """Reference tilt annotation on the orientation panel.

    style="line"  axhline across the full panel
    style="tick"  short segment anchored at the right spine (keeps the
                  depth profiles legible when several markers stack up)
    err may be a scalar (symmetric) or a (lo, hi) pair; show_err=False
    suppresses the band while keeping the line.
    """
    if err is not None and show_err:
        lo, hi = (err, err) if np.isscalar(err) else err
        x0 = 1.0 - tick_frac if style == "tick" else 0.0
        ax.axhspan(value - lo, value + hi, xmin=x0, xmax=1.0,
                   color=color, alpha=0.15, lw=0, zorder=0)
    if style == "line":
        ax.axhline(value, color=color, lw=0.7, ls=ls, zorder=2)
    else:
        ax.plot([1.0 - tick_frac, 1.0], [value, value],
                transform=ax.get_yaxis_transform(), color=color, lw=1.1,
                solid_capstyle="butt", zorder=6)
    ax.annotate(label, xy=(0.985, value), xycoords=("axes fraction", "data"),
                xytext=(0, 2), textcoords="offset points",
                ha="right", va="bottom", color=color, fontsize=fontsize)


# ---- compute the PXR value and assemble the mark list ----------------------
TILT_PXR, TILT_PXR_ERR = tilt_with_stderr(graded_obj, graded_profiles_live)
# posterior alternative (asymmetric):
# TILT_PXR, *TILT_PXR_ERR = tilt_posterior(graded_obj, graded_profiles_live)

TILT_MARKS = [
    dict(value=74, err=None, label="NEXAFS", color="0.45"),   # fill in
    dict(value=None, err=None, label="GIWAXS", color="0.45"),   # fill in
    dict(value=TILT_PXR, err=TILT_PXR_ERR,
         label=r"$\langle\gamma\rangle_{\rho}$", color=C_GRADED),
]


# ---- drop-in replacement for draw_orientation ------------------------------
def draw_orientation(ax, tau_labels=True, marks=TILT_MARKS,
                     mark_style="tick", show_err=True):
    ax.plot(Z_FREE, ALPHA_FREE, color=C_FREE, ls="-", lw=0.9, zorder=3)
    ax.plot(Z_DFT, ALPHA_DFT, color=C_DFT, ls="--", lw=0.9, zorder=3)
    ax.plot(Z_G, ALPHA_G, color=C_GRADED, lw=1, zorder=5)
    for z_tau, txt, ha, dx in ((TAU_VAC, r"$\tau_\mathrm{vac}$", "left", 3),
                               (TOTAL - TAU_SI, r"$\tau_\mathrm{Si}$", "right", -3)):
        ax.axvline(z_tau, color="0.55", lw=0.7, ls=(0, (4, 3)), zorder=1)
        if tau_labels:
            ax.annotate(txt, xy=(z_tau, 0.5), xycoords=("data", "axes fraction"),
                        xytext=(dx, 0), textcoords="offset points",
                        va="top", ha=ha, color="0.4")
    for mk in (marks or []):
        if mk.get("value") is None:
            continue
        draw_tilt_marker(ax, mk["value"], mk.get("err"), mk.get("label", ""),
                         color=mk.get("color", "0.25"),
                         style=mark_style, show_err=show_err)
    ax.set_ylabel(r"$\gamma$ (deg)")
    ax.set_ylim(20, 75)
    ax.set_xlim(0, TOTAL)

In [ ]:
fig = plt.figure(figsize=(3.5, 3.5), dpi=300)
go = fig.add_gridspec(1, 2, width_ratios=[1.55, 1.0], wspace=0.4)
gl = go[0].subgridspec(2, 1, height_ratios=[3.0, 1.0], hspace=0.05)
gr = go[1].subgridspec(2, 1, hspace=0.28)
axR = fig.add_subplot(gl[0])
axres = fig.add_subplot(gl[1], sharex=axR)
axa = fig.add_subplot(gr[0])
axd = fig.add_subplot(gr[1], sharex=axa)

draw_reflectivity(axR)
axR.tick_params(labelbottom=False)
draw_residual(axres)
draw_orientation(axa, marks=None)
axa.tick_params(labelbottom=False)
draw_density(axd)
axd.set_xlabel(r"depth $z\ (\mathrm{\AA})$")

tag(axR, "(a)")
tag(axa, "(b)", -.5)
tag(axd, "(c)", -.5)
fig.align_ylabels([axR, axres])
fig.align_ylabels([axa, axd])
fig.savefig("fig5_option1_nested.pdf", dpi=300, bbox_inches="tight", transparent=True)
fig.savefig("fig5_option1_nested.png", dpi=300, bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Substrate merge shading for the depth-profile panels (b) and (c).
# Call at the END of draw_orientation / draw_density, after set_xlim/set_ylim.
# ---------------------------------------------------------------------------
import numpy as np


def shade_substrate(ax, z0=None, z1=None, color="0.40", alpha_max=0.22,
                    power=2.0, n=80, label=None, label_color="0.35"):
    """Grey gradient that fades in toward the substrate side of the panel.

    The alpha ramps from 0 at z0 to alpha_max at z1, so the film visually
    dissolves into the substrate rather than hitting a wall. This is the
    honest rendering for the graded model, where the buried interface is a
    continuous transition, not a boundary.

    Defaults tie the onset to the buried transition width, z0 = TOTAL - 2*tau_Si,
    so the fade begins where the erf blend becomes non-negligible and reaches
    full strength at the nominal film terminus. power > 1 keeps the onset
    subtle so it does not read as a fourth curve.

    Implemented as adjacent axvspan strips, which span the full axis height
    independent of ylim and survive later autoscaling, unlike imshow.
    """
    if z1 is None:
        z1 = TOTAL
    if z0 is None:
        z0 = TOTAL - 2.0 * TAU_SI
    edges = np.linspace(z0, z1, n + 1)
    alphas = alpha_max * np.linspace(0.0, 1.0, n) ** power
    for x0, x1, a in zip(edges[:-1], edges[1:], alphas):
        ax.axvspan(x0, x1, color=color, alpha=float(a), lw=0, zorder=0)
    if label:
        ax.annotate(label, xy=(z1, 0.03), xycoords=("data", "axes fraction"),
                    xytext=(-2, 0), textcoords="offset points",
                    ha="right", va="bottom", color=label_color)


# usage, appended to the two painters after the axis limits are set:
#   in draw_orientation:  shade_substrate(ax)
#   in draw_density:      shade_substrate(ax, label="Si")

In [ ]:
fig = plt.figure(figsize=(3.5, 3.5), dpi=300)
go = fig.add_gridspec(1, 2, width_ratios=[1.55, 1.0], wspace=0.4)
gl = go[0].subgridspec(2, 1, height_ratios=[3.0, 1.0], hspace=0.05)
gr = go[1].subgridspec(2, 1, hspace=0.28)
axR = fig.add_subplot(gl[0])
axres = fig.add_subplot(gl[1], sharex=axR)
axa = fig.add_subplot(gr[0])
axd = fig.add_subplot(gr[1], sharex=axa)

draw_reflectivity(axR)
axR.tick_params(labelbottom=False)
draw_residual(axres)
draw_orientation(axa, marks=None)
axa.tick_params(labelbottom=False)
shade_substrate(axa)
draw_density(axd)
shade_substrate(axd)
axd.set_xlabel(r"depth $z\ (\mathrm{\AA})$")

tag(axR, "(a)")
tag(axa, "(b)", -.5)
tag(axd, "(c)", -.5)
fig.align_ylabels([axR, axres])
fig.align_ylabels([axa, axd])
fig.savefig("fig4_option1_nested.pdf", dpi=300, bbox_inches="tight", transparent=True)
fig.savefig("fig4_option1_nested.png", dpi=300, bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# "Unwrapped" figure, v10.
#   v10 polish: reduced chi-square moved into the panel (a) legend (color/dash
#   keyed, white-backed) instead of floating over the residual traces; inset
#   decluttered (no ticks, no grid) and re-tied with a locator box; panel
#   letters moved to the interior top-left with a faint pad to clear the rho
#   axis label.
#   (a) reflectivity: FULL data range (no critical-angle clipping), thick solid
#       data traces with shaded +/-1 sigma bands (s blue, p orange, p offset a
#       decade); three model fits overplotted, graded a medium dash on top.
#       A DEDICATED residual strip sits directly beneath (a), sharing the q
#       axis, so fit quality is read on its own linear chi axis instead of
#       being crushed into the log panel. One zoom inset (see ZOOM_WINDOWS)
#       magnifies the high-q window where the models actually separate, tied
#       to the main axis by an indicator box.
#   (b) gamma(z) with a top MLE axis (MLE ticks only, measured from Si).
#   (c) density(z). Depth in Angstrom on the bottom axis.
#
# rcParams are left untouched; every style choice here is local to the axes.
# ---------------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

Q_FULL = np.linspace(float(dataset.s.x.min()), float(dataset.s.x.max()), 2000)

C_SPOL = "#2E6FB0"   # s-pol data (blue)
C_PPOL = "#E38A1E"   # p-pol data (orange)

# Monolayer repeat along depth for the MLE axis (Angstrom).
# SET THIS to your GIWAXS out-of-plane d-spacing; the number below is a stand-in.
D_MONO = 3.34

DPI = 400


# ---- reduced chi-square per model (full data range) ------------------------
def reduced_chi2(mobj, pol=("s", "p")):
    res = []
    for p in pol:
        d = getattr(dataset, p)
        res.append(((d.y - rxr(d.x, mobj.model, p)) / d.y_err) ** 2)
    r = np.concatenate(res)
    try:
        k = len(mobj.varying_parameters())
    except Exception:
        k = 0
    return float(r.sum() / max(r.size - k, 1))


RCHI2 = {m["name"]: reduced_chi2(m["obj"]) for m in MODELS}

# fit line styles: graded = medium dash sitting on the data; slabs recede
_FIT_STYLE = {
    "Graded":    dict(color=C_GRADED, ls=(0, (6, 4)),   lw=1.0, zorder=10),
    "DFT slab":  dict(color="k",      ls=(0, (4, 2)),   lw=0.8, zorder=5),
    "Free slab": dict(color="k",      ls=(0, (1, 1.5)), lw=0.8, zorder=5),
}
_TINY = 1e-30

# high-q windows where the models diverge; each entry becomes one magnified
# inset placed at rect=(x0, y0, w, h) in axes fraction of panel (a). Append a
# second dict, e.g. dict(q=(0.09, 0.16), rect=(0.06, 0.10, 0.40, 0.34)), to
# also zoom the low-q fringe-contrast region.
ZOOM_WINDOWS = [
    dict(q=(0.175, 0.265), rect=(0.60, 0.60, 0.40, 0.40)),
]


# ---------------------------------------------------------------------------
# (a) reflectivity + zoom inset(s)
# ---------------------------------------------------------------------------
def draw_reflectivity_full(ax, p_scale=0.1):
    """Full-range thick data with error bands; fits overplotted."""
    for pol, col, sc in (("s", C_SPOL, 1.0), ("p", C_PPOL, p_scale)):
        d = getattr(dataset, pol)
        o = np.argsort(d.x)
        x, y, e = d.x[o], d.y[o] * sc, d.y_err[o] * sc
        ax.fill_between(x, np.clip(y - e, _TINY, None), y + e, color=col,
                        alpha=0.25, lw=0, zorder=1)
        ax.plot(x, y, "-", color=col, lw=2.0, zorder=2)

    for mdl in MODELS:
        st = _FIT_STYLE[mdl["name"]]
        for pol, sc in (("s", 1.0), ("p", p_scale)):
            ax.plot(Q_FULL, rxr(Q_FULL, mdl["obj"].model, pol) * sc, **st)

    ax.set_yscale("log")
    ax.set_xlim(0, Q_FULL.max())
    ax.set_ylabel("Reflectivity")
    smax = float(dataset.s.y.max())
    pmin = float(dataset.p.y.min()) * p_scale
    ax.set_ylim(pmin * 0.3, smax * 3)   # headroom for the branch labels only

    for pol, sc, col, dy in (("s", 1.0, C_SPOL, 10), ("p", p_scale, C_PPOL, -12)):
        ql = 0.02 if pol == "p" else 0.08
        yq = float(rxr(np.array([ql]), MODELS[0]["obj"].model, pol)[0]) * sc
        ax.annotate(pol, (ql, yq),
                    textcoords="offset points", xytext=(0, dy), ha="center",
                    va="center", color=col)

    handles = [Line2D([], [],
                      label=rf"{m['name']}  $\chi^2_\nu={RCHI2[m['name']]:.2g}$",
                      **{k: v for k, v in _FIT_STYLE[m["name"]].items() if k != "zorder"})
               for m in MODELS]
    ax.legend(handles=handles, loc="lower left", frameon=True,
              facecolor="white", edgecolor="none", framealpha=0.85,
              handlelength=1.8, labelspacing=0.35, borderpad=0.3)


def add_zoom_insets(ax, p_scale=0.1, windows=ZOOM_WINDOWS, npts=400):
    """Magnified log-R inset(s) over model-discriminating q windows.

    Each inset replots the bands, data, and three fits inside the window,
    autoscales y to the enclosed data, and is tied to the parent axis by an
    indicator box via indicate_inset_zoom.
    """
    for spec in windows:
        q0, q1 = spec["q"]
        axins = ax.inset_axes(spec["rect"])
        lo, hi = np.inf, -np.inf
        for pol, col, sc in (("s", C_SPOL, 1.0), ("p", C_PPOL, p_scale)):
            d = getattr(dataset, pol)
            o = np.argsort(d.x)
            x, y, e = d.x[o], d.y[o] * sc, d.y_err[o] * sc
            m = (x >= q0) & (x <= q1)
            if not m.any():
                continue
            axins.fill_between(x[m], np.clip(y[m] - e[m], _TINY, None), y[m] + e[m],
                               color=col, alpha=0.25, lw=0, zorder=1)
            axins.plot(x[m], y[m], "-", color=col, lw=1.4, zorder=2)
            lo = min(lo, float(np.nanmin(np.clip(y[m] - e[m], _TINY, None))))
            hi = max(hi, float(np.nanmax(y[m] + e[m])))
        qq = np.linspace(q0, q1, npts)
        for mdl in MODELS:
            st = _FIT_STYLE[mdl["name"]]
            for pol, sc in (("s", 1.0), ("p", p_scale)):
                axins.plot(qq, rxr(qq, mdl["obj"].model, pol) * sc, **st)
        axins.set_yscale("log")
        axins.set_xlim(q0, q1)
        if np.isfinite(lo) and np.isfinite(hi):
            axins.set_ylim(lo * 0.5, hi * 2.0)
        axins.set_xticks([q0, q1])
        axins.set_xticklabels([])
        axins.set_yticklabels([])
        axins.tick_params(length=2)
        axins.grid(False)                       # magnified box stays uncluttered
        for sp in axins.spines.values():
            sp.set_linewidth(0.8)
        # locator box ties the inset to its q window; the auto connector lines
        # clip to the figure (not the axes), so hide them to avoid bleed below.
        ind = ax.indicate_inset_zoom(axins, edgecolor="0.35", lw=0.6, alpha=0.9)
        try:
            conns = ind.connectors
        except AttributeError:
            _, conns = ind
        for c in conns:
            c.set_visible(False)


def draw_residual_panel(ax, pol="p", clip_pct=98):
    """Dedicated linear-chi strip below (a), sharing the q axis.

    chi = (data - model)/sigma for each model, in its own color/dash. The
    graded curve hugging zero while the slabs swing is the direct visual
    statement of the fit improvement. Reduced chi-square is annotated per
    model in the top-right corner.
    """
    d = getattr(dataset, pol)
    o = np.argsort(d.x)
    x = d.x[o]
    ax.axhline(0.0, color="0.6", lw=0.6, zorder=1)
    hi = 0.0
    for mdl in MODELS:
        chi = ((d.y - rxr(d.x, mdl["obj"].model, pol)) / d.y_err)[o]
        st = _FIT_STYLE[mdl["name"]]
        ax.plot(x, chi, color=st["color"], ls=st["ls"], lw=0.8,
                zorder=st["zorder"])
        hi = max(hi, float(np.nanpercentile(np.abs(chi), clip_pct)))
    lim = max(float(np.ceil(hi)), 1.0)
    ax.set_xlim(0, Q_FULL.max())
    ax.set_ylim(-lim, lim)
    ax.set_yticks([-lim, 0, lim])
    ax.set_ylabel(rf"$\chi_{pol}$")
    ax.set_xlabel(r"$q\ (\mathrm{\AA}^{-1})$")


# ---------------------------------------------------------------------------
# (b) orientation + (c) density
# ---------------------------------------------------------------------------
def draw_tilt_marker(ax, value, err=None, label="", color="0.25",
                     style="tick", show_err=True, tick_frac=0.14,
                     ls=(0, (1, 1.5))):
    if err is not None and show_err:
        lo, hi = (err, err) if np.isscalar(err) else err
        x0 = 1.0 - tick_frac if style == "tick" else 0.0
        ax.axhspan(value - lo, value + hi, xmin=x0, xmax=1.0,
                   color=color, alpha=0.15, lw=0, zorder=0)
    if style == "line":
        ax.axhline(value, color=color, lw=0.7, ls=ls, zorder=2)
    else:
        ax.plot([1.0 - tick_frac, 1.0], [value, value],
                transform=ax.get_yaxis_transform(), color=color, lw=1.1,
                solid_capstyle="butt", zorder=6)
    y0, y1 = ax.get_ylim()
    near_top = value > y1 - 0.10 * (y1 - y0)
    ax.annotate(label, xy=(0.985, value), xycoords=("axes fraction", "data"),
                xytext=(0, -8 if near_top else 2), textcoords="offset points",
                ha="right", va="top" if near_top else "bottom", color=color)


def draw_orientation(ax, tau_labels=True, marks=TILT_MARKS, show_marks=False,
                     mark_style="tick", show_err=True):
    ax.plot(Z_FREE, ALPHA_FREE, color=C_FREE, ls="-", lw=0.9, zorder=3)
    ax.plot(Z_DFT, ALPHA_DFT, color=C_DFT, ls="--", lw=0.9, zorder=3)
    ax.plot(Z_G, ALPHA_G, color=C_GRADED, lw=1.4, zorder=5)
    for z_tau, txt, ha, dx in ((TAU_VAC, r"$\tau_\mathrm{vac}$", "left", 3),
                               (TOTAL - TAU_SI, r"$\tau_\mathrm{Si}$", "right", -3)):
        ax.axvline(z_tau, color="0.55", lw=0.7, ls=(0, (4, 3)), zorder=1)
        if tau_labels:
            ax.annotate(txt, xy=(z_tau, 0.5), xycoords=("data", "axes fraction"),
                        xytext=(dx, 0), textcoords="offset points",
                        va="top", ha=ha, color="0.4")
    if show_marks:
        for mk in (marks or []):
            if mk.get("value") is None:
                continue
            draw_tilt_marker(ax, mk["value"], mk.get("err"), mk.get("label", ""),
                             color=mk.get("color", "0.25"),
                             style=mark_style, show_err=show_err)
    ax.set_ylabel(r"$\gamma$ (deg)")
    ax.set_ylim(20, 90)
    ax.set_xlim(0, TOTAL)


# ---------------------------------------------------------------------------
# assemble
# ---------------------------------------------------------------------------
def _panel_label(ax, s):
    # inside the top-left corner with a faint white pad; robust to the long
    # rho axis label that the old outside-left placement collided with.
    ax.text(0.03, 0.96, s, transform=ax.transAxes, ha="left", va="top",
            bbox=dict(boxstyle="square,pad=0.15", fc="white", ec="none",
                      alpha=0.7))


def build_unwrapped_figure(figsize=(3.5, 6.6), p_scale=0.1, d_ml=D_MONO,
                           zoom=True):
    fig = plt.figure(figsize=figsize, dpi=DPI)
    outer = fig.add_gridspec(2, 1, height_ratios=[2.9, 2.2], hspace=0.42)
    top = outer[0].subgridspec(2, 1, height_ratios=[3.0, 1.0], hspace=0.05)
    ax_r = fig.add_subplot(top[0])
    ax_res = fig.add_subplot(top[1], sharex=ax_r)     # dedicated residual strip
    bot = outer[1].subgridspec(2, 1, height_ratios=[1.0, 1.0], hspace=0.06)
    ax_o = fig.add_subplot(bot[0])
    ax_d = fig.add_subplot(bot[1], sharex=ax_o)

    draw_reflectivity_full(ax_r, p_scale=p_scale)
    if zoom:
        add_zoom_insets(ax_r, p_scale=p_scale)
    ax_r.tick_params(labelbottom=False)
    draw_residual_panel(ax_res)

    draw_orientation(ax_o)
    draw_density(ax_d)
    shade_substrate(ax_o)
    shade_substrate(ax_d, label="Si")

    # MLE top axis (from Si). Turn OFF the primary top ticks so only MLE shows.
    ax_o.tick_params(top=False, which="both")
    secx = ax_o.secondary_xaxis("top", functions=(lambda z: (TOTAL - z) / d_ml,
                                                  lambda n: TOTAL - n * d_ml))
    secx.set_xlabel("MLE")

    ax_o.tick_params(labelbottom=False)
    ax_o.set_xlabel("")
    ax_d.set_xlabel(r"depth $z\ (\mathrm{\AA})$")

    _panel_label(ax_r, "(a)")
    _panel_label(ax_o, "(b)")
    _panel_label(ax_d, "(c)")
    fig.align_ylabels([ax_r, ax_res, ax_o, ax_d])
    return fig, (ax_r, ax_res, ax_o, ax_d)


fig, axes = build_unwrapped_figure()
# fig.savefig("figure_unwrapped.pdf", bbox_inches="tight")


In [ ]:
fig = plt.figure(figsize=(3.4, 7.0), dpi=300)
gs = fig.add_gridspec(4, 1, height_ratios=[3.0, 1.1, 2.2, 2.2], hspace=0.18)

axR = fig.add_subplot(gs[0])
axres = fig.add_subplot(gs[1], sharex=axR)   # shares q with (1)
axa = fig.add_subplot(gs[2])                 # depth axis begins here
axd = fig.add_subplot(gs[3], sharex=axa)

draw_reflectivity(axR); axR.tick_params(labelbottom=False)
draw_residual(axres)
draw_orientation(axa); axa.tick_params(labelbottom=False)
draw_density(axd); axd.set_xlabel(r"depth $z\ (\mathrm{\AA})$")

tag(axR, "(1)", -.1); tag(axres, "(2)", -.1); tag(axa, "(3)", -.1); tag(axd, "(4)", -.1)
fig.align_ylabels([axR, axres, axa, axd])
fig.savefig("fig5_option2_stack.png", dpi=300, bbox_inches="tight", transparent=True)
plt.show()

## Refloxide comparison (new)

Adds the refloxide multi-energy UniTensorSLD fit (`refloxide/examples/real_data_repl.py`,
seeded from the free-tensor model's own geometry) as a fourth model alongside
Graded/DFT slab/Free slab. Uses `utils.refloxide_view` -- a small read-only
adapter (NOT a full pyref API replacement) that lets this repo's own
`rxr`/`slab_film_profiles` helpers treat the refloxide fit exactly like the
three pyref-based ones above, since it exposes the same `model.pol = ...;
model(q)` calling convention and the same `vacuum | surface | bulk |
interface | oxide | substrate` slab ordering.

In [ ]:
from utils import models_root
from utils.refloxide_view import load_refloxide_fit, objective_view_at

REFLOXIDE_FIT_PATH = models_root / "xrr/znpc/refloxide/refit-refloxide.pkl"
if not REFLOXIDE_FIT_PATH.exists():
    raise FileNotFoundError(
        f"missing {REFLOXIDE_FIT_PATH} -- run refloxide/examples/real_data_repl.py "
        "(through the CurveFitter.fit cell) first to produce this pickle"
    )

refloxide_objective = load_refloxide_fit(REFLOXIDE_FIT_PATH)
refloxide_obj = objective_view_at(refloxide_objective, ENERGY)
print(f"loaded refloxide fit: {len(refloxide_obj.varying_parameters())} varying parameters")

In [ ]:
C_REFLOXIDE = "tab:green"

refloxide_z, refloxide_alpha, refloxide_rho = slab_film_profiles(refloxide_obj.model)
refloxide_rms = rms_residual(refloxide_obj.model)

fig = plt.figure(figsize=(3.5, 3.5), dpi=300)
go = fig.add_gridspec(1, 2, width_ratios=[1.55, 1.0], wspace=0.4)
gl = go[0].subgridspec(2, 1, height_ratios=[3.0, 1.0], hspace=0.05)
gr = go[1].subgridspec(2, 1, hspace=0.28)
axR = fig.add_subplot(gl[0])
axres = fig.add_subplot(gl[1], sharex=axR)
axa = fig.add_subplot(gr[0])
axd = fig.add_subplot(gr[1], sharex=axa)

draw_reflectivity(axR)
for pol in ("s", "p"):
    axR.plot(
        Q,
        rxr(Q, refloxide_obj.model, pol),
        color=C_REFLOXIDE,
        lw=1,
        zorder=6,
        label="Refloxide" if pol == "s" else None,
    )
axR.legend(loc="lower left", handlelength=1)
axR.tick_params(labelbottom=False)

draw_residual(axres)
d = dataset.p
m = d.x >= QMIN
chi_refloxide = (d.y[m] - rxr(d.x[m], refloxide_obj.model, "p")) / d.y_err[m]
axres.plot(d.x[m], chi_refloxide, color=C_REFLOXIDE, lw=0.6, zorder=6)

draw_orientation(axa, marks=None)
axa.plot(refloxide_z, refloxide_alpha, color=C_REFLOXIDE, lw=1, zorder=6)
axa.tick_params(labelbottom=False)

draw_density(axd)
axd.plot(refloxide_z, refloxide_rho, color=C_REFLOXIDE, lw=1, zorder=6)
axd.set_xlabel(r"depth $z\ (\mathrm{\AA})$")

tag(axR, "(a)")
tag(axa, "(b)", -0.5)
tag(axd, "(c)", -0.5)
fig.align_ylabels([axR, axres])
fig.align_ylabels([axa, axd])
fig.savefig("figures/fig5_with_refloxide.png", dpi=300, bbox_inches="tight", transparent=True)
plt.show()

print(f"RMS residual (s+p): Refloxide={refloxide_rms:.2f}")